# Questionnaire — configured by one file

This notebook runs a **questionnaire** and saves the answers. Like the other
whispy UIs it is **config-driven**: you describe the questions in
[`configs/questionnaires/questionnaire_general.yml`](../configs/questionnaires/questionnaire_general.yml)
and the shared look comes from [`configs/design.yml`](../configs/design.yml).
You should not need to change any Python below — just edit the YAML.

That config file has two parts:

| block in the YAML | configures |
|---|---|
| `ui:` | layout & sizing (window size, fonts, response-box widths, spacing). Colors are inherited from `design.yml`. |
| `questionnaire:` | the content — a list of **sections**, each holding a list of **questions**. |

**Supported question types:** `text` (one line), `text_box` (multi-line),
`numeric` (numbers only), `single_choice` (radio buttons, with an optional
free-text "other" field) and `multiple_choice` (checkboxes).

A question can also be shown conditionally with a `depends_on` block, so it only
appears once a controlling question has a matching answer.

If you haven't installed the project yet, run this from the repo root:
`pip install -e .`

In [ ]:
import whispy
from pathlib import Path
from whispy.ui import Questionnaire
from whispy.utils import read_config

config_path = Path('..') / 'configs' / 'questionnaires' / 'questionnaire_general.yml'   # <- the file you edit
cfg = read_config(config_path)
print('sections:', [s['section'] for s in cfg['questionnaire']])

## See your questionnaire at a glance

Open [`configs/questionnaires/questionnaire_general.yml`](../configs/questionnaires/questionnaire_general.yml)
to edit every question (it has inline comments). The cell below prints the
current structure — each section, then its questions with their `type`, whether
they are `required`, and any choice `options` — so you can review the whole
survey without leaving the notebook.

In [2]:
for s_i, section in enumerate(cfg['questionnaire'], start=1):
    print(f"{s_i}. {section['section']}  —  {section.get('prompt', '')}")
    for q_i, q in enumerate(section.get('questions', []), start=1):
        flag = 'required' if q.get('required') else 'optional'
        line = f"   {s_i}.{q_i} [{q['type']}, {flag}]  {q['question']}: {q.get('prompt', '')}"
        if q.get('options'):
            line += f"   options={q['options']}"
        if q.get('other_question'):
            line += "   (+ free-text 'other')"
        print(line)

1. general_information  —  General information about yourself
   1.1 [text, required]  headphones: What headphones are you using for the test?
   1.2 [single_choice, required]  gender: What gender do you identify with most?   options=['Male', 'Female', 'Diverse', 'Prefer not to disclose']
   1.3 [numeric, required]  birth_year: What year were you born?
   1.4 [text, required]  native_language: What is your native language? You can name multiple.
   1.5 [single_choice, required]  language_proficiency: Please rate your current level of proficiency in the English language.   options=['A1 - Breakthrough Basic User', 'A2 - Waystage Basic User', 'B1 - Threshold Independent User', 'B2 - Vantage Independent User', 'C1 - Advanced Proficient User', 'C2 - Mastery Proficient User', 'Native speaker', 'Other']   (+ free-text 'other')
   1.6 [single_choice, required]  hearing_impairment: Do you have a medically diagnosed hearing impairment?   options=['no', 'yes, pertains to the right ear', 'yes, per

## How each question type behaves

- **`text`** — a single-line text box; answered when non-empty.
- **`text_box`** — a multi-line box (height set by `ui.text_box_number_of_lines`).
- **`numeric`** — a box that only accepts numbers; the answer comes back as a float.
- **`single_choice`** — radio buttons (pick one). Add an `other_question` block to reveal a free-text field when a specific option is selected (see `favorite_animal` in the example config).
- **`multiple_choice`** — checkboxes (pick any number); the answer is a list.

Mark a question `required: true` to force an answer. When the participant clicks
**Continue**, any unanswered required questions are listed in a popup and the
window stays open until they are filled in. Set `ui.enumerate: true` to number
the sections and questions automatically.

## Run the questionnaire

The cell below opens the questionnaire window and **blocks** until the
participant finishes and clicks **Continue**. As with the other whispy UIs,
`Continue` only releases control once every required question is answered; the
window then stays open until we explicitly `close()` it.

> Tip: while developing, pass `debug=True` to `Questionnaire(...)` to enable the
> native close button and skip the required-answer check.

In [ ]:
questionnaire = Questionnaire(questionnaire=config_path, blocking=True)
results = questionnaire.get_results()
questionnaire.close()
results

## The results

`get_results()` returns a `pandas.DataFrame` with **one row per question** and
the columns `section, question, prompt, type, required, answer`. For
`multiple_choice` the `answer` is a list of the selected options; for a
`single_choice` with an "other" field it is the typed text when that option was
chosen.

In [ ]:
print('answered rows:', len(results))
results

## Save the results

Write the answers to a timestamped CSV in a `results/` folder (created next to
this notebook), so each participant's responses are kept for later analysis.

In [ ]:
from datetime import datetime

results_dir = Path('results')          # CSVs are written next to this notebook
results_dir.mkdir(exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
results_path = results_dir / f'questionnaire_{timestamp}.csv'
results.to_csv(results_path, index=False)
print('saved results to', results_path.resolve())